In [10]:
import xarray as xr
import rioxarray
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score
from scipy.ndimage import median_filter, minimum_filter, maximum_filter, uniform_filter
import lightgbm as lgb 
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import joblib

# Load input data and trained model

In [11]:
def predict(region: str, year: int):
    # Load model payload
    payload = joblib.load(f'trained-{region.lower()}-2021.joblib')
    
    # Extract the list of models instead of a single model
    clfs = payload['clf']
    user_attrs = payload['user_attrs']
    
    # Safely compute micro-averaged precision and recall
    precision = (np.array(user_attrs['precisions']) * np.array(user_attrs['val_sizes'])).sum() / sum(user_attrs["val_sizes"])
    recall = (np.array(user_attrs['recalls']) * np.array(user_attrs['val_sizes'])).sum() / sum(user_attrs["val_sizes"])

    # Load input data
    df = pd.read_parquet(f"../data/silver/ghana-{year}-df.parquet")

    # Extract features
    features = ['blue', "green", "red", "rededge", "B06", "B07", "nir", "B8A", "swir11", "swir22"]
    features += ['scl', "region"]
    features += ['tci', 'ndvi', 'evi', 'evi2', 'ndre_b5', 'ndre_b6', 'ndre_b7', 'gndvi', 'ndwi']
    features += ['valid_observations']
    features += ['cloud_confidence']
    
    # Filter for the target region
    region_mask = df.region.apply(lambda x: x.lower()) == region.lower()
    X = df.loc[region_mask, features]

    # Run inference for EACH model in our ensemble
    # all_yhats will have shape: (n_models, n_samples)
    all_yhats = np.array([clf.predict(X) for clf in clfs])

    return all_yhats, precision, recall

# Real area from Cocoa area estimation

My issue is that I'm dealing with 2 back-to-back area estimation.
Hence, for each region, I have

1. The real cocoa area
2. The ETHZ's cocoa area estimation
3. My cocoa area estimation

First, I need to figure out how to estimate the real cocoa area from the ETHZ's cocoa area estimation.

## Our estimate

The ETHZ's estimated area are the best we'll get.
We make the assumption that the ETHZ's estimate is correct, i.e

$$Â^{corrected}_{ETHZ} = A_{true}$$

From them, we'll compare our estimate, i.e.

$$A_{true} = Â^{corrected}_{ETHZ} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$
$$\rightarrow Â^{corrected}_{LGBM} = A_{LGBM} * \frac{Precision^{Val}_{LGBM}}{Recall^{Val}_{LGBM}}$$

We'll measure how far off $Â^{corrected}_{ETHZ}$ we are

In [12]:
def compute_corrected_area_proportion(y, precision, recall):
    area_proportion = y.sum() / len(y)
    return area_proportion * precision / recall

In [14]:
def print_our_corrected_area_proportion(region: str, year: int, ci_level: float = 0.95, n_bootstraps: int = 1000) -> float:
    """Print the Corrected Area Proportion (CAP) of our prediction with a Bootstrapped Confidence Interval."""
    # Get the prediction matrix and metrics
    all_yhats, precision, recall = predict(region, year)

    n_samples = all_yhats.shape[1]
    bootstrapped_proportions = []

    # Set random seed for reproducible bootstrapping
    np.random.seed(42)
    
    # The Bootstrap Loop
    for _ in range(n_bootstraps):
        # Draw random indices with replacement from the entire region dataset
        boot_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Compute the CAP for every model on this specific bootstrapped sample
        for yhats in all_yhats:
            yhat_boot = yhats[boot_indices]
            ap = compute_corrected_area_proportion(yhat_boot, precision, recall)
            bootstrapped_proportions.append(ap)

    # Convert to array for percentile math
    bootstrapped_proportions = np.array(bootstrapped_proportions)
    
    mean_ap = np.mean(bootstrapped_proportions)
    
    # Calculate bounds based on the specified CI level
    alpha = 1.0 - ci_level
    lower_bound = np.percentile(bootstrapped_proportions, (alpha / 2) * 100)
    upper_bound = np.percentile(bootstrapped_proportions, (1 - (alpha / 2)) * 100)

    # Format the output bounds
    ap_str = f"{mean_ap * 100:.3f}% (95% CI: [{lower_bound * 100:.3f}%, {upper_bound * 100:.3f}%])"

    print(f"{region.title()}'s entire region ({year}): Our cocoa area proportion (corrected): {ap_str}")
    
print_our_corrected_area_proportion('ashanti', 2022)
print_our_corrected_area_proportion('western north', 2022)
print_our_corrected_area_proportion('western', 2022)
print_our_corrected_area_proportion('ahafo', 2022)
print_our_corrected_area_proportion('central', 2022)

Ashanti's entire region (2022): Our cocoa area proportion (corrected): 65.416% (95% CI: [63.365%, 67.163%])
Western North's entire region (2022): Our cocoa area proportion (corrected): 97.624% (95% CI: [97.219%, 97.929%])
Western's entire region (2022): Our cocoa area proportion (corrected): 93.786% (95% CI: [93.465%, 94.091%])
Ahafo's entire region (2022): Our cocoa area proportion (corrected): 87.195% (95% CI: [86.290%, 88.074%])
Central's entire region (2022): Our cocoa area proportion (corrected): 80.028% (95% CI: [66.943%, 88.255%])


In [ ]:
def print_our_corrected_area_proportion(region: str, year: int, ci_level: float = 0.95, n_bootstraps: int = 1000) -> float:
    """Print the Corrected Area Proportion (CAP) of our prediction with a Bootstrapped Confidence Interval."""
    # Get the prediction matrix and metrics
    all_yhats, precision, recall = predict(region, year)

    n_samples = all_yhats.shape[1]
    bootstrapped_proportions = []

    # Set random seed for reproducible bootstrapping
    np.random.seed(42)
    
    # The Bootstrap Loop
    for _ in range(n_bootstraps):
        # Draw random indices with replacement from the entire region dataset
        boot_indices = np.random.choice(n_samples, size=n_samples, replace=True)
        
        # Compute the CAP for every model on this specific bootstrapped sample
        for yhats in all_yhats:
            yhat_boot = yhats[boot_indices]
            ap = compute_corrected_area_proportion(yhat_boot, precision, recall)
            bootstrapped_proportions.append(ap)

    # Convert to array for percentile math
    bootstrapped_proportions = np.array(bootstrapped_proportions)
    
    mean_ap = np.mean(bootstrapped_proportions)
    
    # Calculate bounds based on the specified CI level
    alpha = 1.0 - ci_level
    lower_bound = np.percentile(bootstrapped_proportions, (alpha / 2) * 100)
    upper_bound = np.percentile(bootstrapped_proportions, (1 - (alpha / 2)) * 100)

    # Format the output bounds
    ap_str = f"{mean_ap * 100:.3f}% (95% CI: [{lower_bound * 100:.3f}%, {upper_bound * 100:.3f}%])"

    print(f"{region.title()}'s entire region ({year}): Our cocoa area proportion (corrected): {ap_str}")
    
print_our_corrected_area_proportion('ashanti', 2022)
print_our_corrected_area_proportion('western north', 2022)
print_our_corrected_area_proportion('western', 2022)
print_our_corrected_area_proportion('ahafo', 2022)
print_our_corrected_area_proportion('central', 2022)